# Todas_las_imagenes

Notebook para abrir y visualizar todos los especimenes disponibles, comparando la imagen H&E (`.mrxs`) con una pseudo-RGB generada desde la imagen HSI (`.hdr`).

La ruta HSI de `SB012` se ha corregido porque el archivo real esta dentro de la carpeta `SB012_M1_002`.

In [ ]:
print("SB013")

In [ ]:
from pathlib import Path
import gc

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

plt.rcParams['figure.dpi'] = 120

In [ ]:
print("SB012")

In [ ]:
SPECIMENS = [
    {
        'id': 'SB012',
        'he_path': Path(r'Datos\SB012\SB012\H&E\SB012\SB012.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr'),
    },
    {
        'id': 'SB013',
        'he_path': Path(r'Datos\SB013\SB013\H&E\SB013.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr'),
    },
    {
        'id': 'SB017',
        'he_path': Path(r'Datos\SB017_uomo\SB017_uomo\H&E\SB017\SB017.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr'),
    },
    {
        'id': 'SB018',
        'he_path': Path(r'Datos\SB018\SB018\H&E\SB018\SB018.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr'),
    },
    {
        'id': 'SB019',
        'he_path': Path(r'Datos\SB019\SB019\H&E\SB019\SB019.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr'),
    },
    {
    'id': 'SB020',
    'he_path': Path(r'Datos\SB020\SB020\H&E\SB020\SB020.mrxs'),
    'hsi_hdr_path': Path(r'Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr'),
    },
]

for specimen in SPECIMENS:
    print(specimen['id'])
    print('  H&E:', specimen['he_path'], '| existe:', specimen['he_path'].exists())
    print('  HSI:', specimen['hsi_hdr_path'], '| existe:', specimen['hsi_hdr_path'].exists())

In [ ]:
def read_he_preview(he_path):
    """Abre la H&E igual que en Datos/SB013/1_SB013.ipynb: PIL.Image.open sobre el .mrxs."""
    he_path = Path(he_path)
    if not he_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {he_path}')

    img = Image.open(he_path)
    img_format = img.format
    img = img.convert('RGB')
    rgb = np.asarray(img, dtype=np.uint8)
    info = {
        'format': img_format,
        'shape': rgb.shape,
        'mode': img.mode,
    }
    return rgb, info


def robust_normalize(channel, p_low=2, p_high=98):
    channel = np.asarray(channel, dtype=np.float32)
    finite_mask = np.isfinite(channel)

    if not np.any(finite_mask):
        return np.zeros_like(channel, dtype=np.float32)

    lo = np.nanpercentile(channel, p_low)
    hi = np.nanpercentile(channel, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(channel, dtype=np.float32)

    normalized = (channel - lo) / (hi - lo)
    normalized = np.clip(normalized, 0, 1)
    return np.nan_to_num(normalized, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def parse_envi_header(hdr_path):
    hdr_path = Path(hdr_path)
    text = hdr_path.read_text(encoding='utf-8', errors='ignore')
    metadata = {}

    current_key = None
    current_value = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.upper() == 'ENVI':
            continue

        if current_key is not None:
            current_value.append(line)
            if '}' in line:
                metadata[current_key] = ' '.join(current_value)
                current_key = None
                current_value = []
            continue

        if '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()

        if value.startswith('{') and '}' not in value:
            current_key = key
            current_value = [value]
        else:
            metadata[key] = value

    return metadata


def parse_wavelengths(metadata):
    values = metadata.get('wavelength')
    if values is None:
        return None

    values = values.strip().strip('{}')
    return np.array([float(v.strip()) for v in values.split(',') if v.strip()], dtype=np.float32)


def envi_dtype(metadata):
    data_type = int(metadata['data type'])
    byte_order = int(metadata.get('byte order', 0))
    endian = '<' if byte_order == 0 else '>'
    dtype_map = {
        1: 'u1',
        2: 'i2',
        3: 'i4',
        4: 'f4',
        5: 'f8',
        12: 'u2',
        13: 'u4',
        14: 'i8',
        15: 'u8',
    }
    if data_type not in dtype_map:
        raise ValueError(f'Tipo de dato ENVI no soportado: {data_type}')
    return np.dtype(endian + dtype_map[data_type])


def find_envi_data_path(hdr_path, metadata):
    hdr_path = Path(hdr_path)
    candidates = []

    for key in ('data file', 'file name', 'image'):
        if key in metadata:
            value = metadata[key].strip().strip('{}').strip('"')
            candidates.append(hdr_path.parent / value)

    candidates.extend([
        hdr_path.with_suffix(''),
        hdr_path.parent / hdr_path.stem.replace('_edu', ''),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f'No se encontro el binario ENVI asociado a {hdr_path.name}')


def read_envi_band(hdr_path, metadata, band_idx):
    data_path = find_envi_data_path(hdr_path, metadata)
    samples = int(metadata['samples'])
    lines = int(metadata['lines'])
    bands = int(metadata['bands'])
    offset = int(metadata.get('header offset', 0))
    interleave = metadata.get('interleave', 'bsq').strip().lower()
    dtype = envi_dtype(metadata)

    if interleave == 'bsq':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(bands, lines, samples))
        return np.asarray(cube[band_idx], dtype=np.float32)
    if interleave == 'bil':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, bands, samples))
        return np.asarray(cube[:, band_idx, :], dtype=np.float32)
    if interleave == 'bip':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, samples, bands))
        return np.asarray(cube[:, :, band_idx], dtype=np.float32)

    raise ValueError(f'Interleave ENVI no soportado: {interleave}')


def hsi_pseudo_rgb_from_path(hdr_path, r_nm=650, g_nm=550, b_nm=450):
    """Genera una pseudo-RGB de una HSI leyendo solo tres bandas del .hdr."""
    hdr_path = Path(hdr_path)
    if not hdr_path.exists():
        raise FileNotFoundError(f'No existe la HSI: {hdr_path}')

    metadata = parse_envi_header(hdr_path)
    wavelengths = parse_wavelengths(metadata)

    if wavelengths is None:
        band_count = int(metadata['bands'])
        r_idx = min(band_count - 1, int(round(0.72 * (band_count - 1))))
        g_idx = min(band_count - 1, int(round(0.50 * (band_count - 1))))
        b_idx = min(band_count - 1, int(round(0.28 * (band_count - 1))))
        band_info = {'r_idx': r_idx, 'g_idx': g_idx, 'b_idx': b_idx}
    else:
        r_idx = int(np.argmin(np.abs(wavelengths - r_nm)))
        g_idx = int(np.argmin(np.abs(wavelengths - g_nm)))
        b_idx = int(np.argmin(np.abs(wavelengths - b_nm)))
        band_info = {
            'r_idx': r_idx,
            'r_nm': float(wavelengths[r_idx]),
            'g_idx': g_idx,
            'g_nm': float(wavelengths[g_idx]),
            'b_idx': b_idx,
            'b_nm': float(wavelengths[b_idx]),
        }

    rgb = np.stack(
        [
            robust_normalize(read_envi_band(hdr_path, metadata, r_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, g_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, b_idx)),
        ],
        axis=-1,
    )
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)
    rgb = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

    info = {
        'shape': rgb.shape,
        **band_info,
    }
    return rgb, info


In [ ]:
def show_specimen(specimen):
    specimen_id = specimen['id']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

    try:
        he_rgb, he_info = read_he_preview(specimen['he_path'])
        axes[0].imshow(he_rgb)
        axes[0].set_title(
            f'{specimen_id} - H&E preview 2D\n'
            f"{he_info['format']} | {he_info['shape'][1]}x{he_info['shape'][0]} px"
        )
    except Exception as exc:
        axes[0].text(0.5, 0.5, str(exc), ha='center', va='center', wrap=True)
        axes[0].set_title(f'{specimen_id} - H&E no abierta')

    try:
        hsi_rgb, hsi_info = hsi_pseudo_rgb_from_path(specimen['hsi_hdr_path'])
        axes[1].imshow(hsi_rgb)
        if {'r_nm', 'g_nm', 'b_nm'}.issubset(hsi_info):
            band_text = f"R={hsi_info['r_nm']:.1f} nm, G={hsi_info['g_nm']:.1f} nm, B={hsi_info['b_nm']:.1f} nm"
        else:
            band_text = f"bandas {hsi_info['r_idx']}, {hsi_info['g_idx']}, {hsi_info['b_idx']}"
        axes[1].set_title(
            f'{specimen_id} - HSI pseudo-RGB\n'
            f"{band_text} | {hsi_info['shape'][1]}x{hsi_info['shape'][0]} px"
        )
    except Exception as exc:
        axes[1].text(0.5, 0.5, str(exc), ha='center', va='center', wrap=True)
        axes[1].set_title(f'{specimen_id} - HSI no abierta')

    for ax in axes:
        ax.axis('off')

    plt.show()
    gc.collect()


def show_all_specimens(specimens=SPECIMENS):
    for specimen in specimens:
        print(f"\n=== {specimen['id']} ===")
        show_specimen(specimen)


In [ ]:
show_all_specimens(SPECIMENS)